# all-reduce-grad-sync — ex2: skip-sync optimization — only all_reduce non-None grads

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `all-reduce-grad-sync`. Running the final beacon cell reports progress against the `Distributed: all_reduce grad sync` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Distributed: all_reduce grad sync` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`all-reduce-grad-sync`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "all-reduce-grad-sync"
DD_SUBTOPIC = "Distributed: all_reduce grad sync"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Skip-sync optimization — only all_reduce non-None grads

Ex1 looped every parameter and all_reduced its `.grad`. In real training, some parameters get NO gradient on this step:

- Frozen layers (`requires_grad=False`).
- Sparse models where this batch didn't touch certain heads.
- Embedding tables when none of this batch's tokens hit them.

Their `.grad` is `None`. Calling `dist.all_reduce(None, ...)` either crashes or silently sends a zero tensor — wasted bandwidth. The skip-sync optimization:

```python
for p in model.parameters():
    if p.grad is None:
        continue
    dist.all_reduce(p.grad, op=dist.ReduceOp.SUM)
    p.grad /= world_size
```

**Subtlety: rank consistency.** Every rank must agree on WHICH parameters to skip — if rank 0 skips `embedding.weight` but rank 1 does not, you deadlock (rank 1 hangs waiting for rank 0's all_reduce that never comes). In practice every rank runs the same code over the same model graph, so the `is None` check returns the same answer everywhere. If your grads diverge across ranks before sync, that's a bug ABOVE this layer.

**Why not iterate `model.named_parameters()` instead.** Same answer; `.parameters()` is the canonical form and you don't need the names for this op.

### Exercise 2 — skip-sync optimization — only all_reduce non-None grads

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Bloom level: Apply
> LO: Apply the `if p.grad is None: continue` skip-sync optimization before `dist.all_reduce(p.grad, SUM)` + divide-by-world_size, verified by running on a model with one frozen layer.
> Keywords: DDP, grad-sync, skip-sync, frozen-layers, sparse-grads
> ```

**KCs targeted:** `skip-none-grads-in-sync-loop`, `all-reduce-mean-divide-after-skip`

Implement `ex2_grad_sync_skip_none(rank, world_size, dist_module, model)`. The frozen-layer-aware grad sync:

1. Loop `for p in model.parameters()`.
2. **If `p.grad is None`: `continue`.** This skips parameters that didn't receive a gradient on this step (frozen layers, sparse heads, etc.).
3. Otherwise, all-reduce the grad: `dist_module.all_reduce(p.grad, op=dist_module.ReduceOp.SUM)`.
4. Divide by `world_size` in-place: `p.grad /= world_size`.
5. Return the integer COUNT of parameters that were synced (i.e. had non-None grads). The test asserts this count matches expectations.

Important: every rank runs this same loop on the same model graph, so the `is None` decision is consistent across ranks — no risk of deadlock.

Input: `rank`, `world_size` — ints; `dist_module` — torch.distributed or mock; `model` — `nn.Module` with some parameters' `.grad` set, some left as `None`.
Output: `int` — count of parameters synced (i.e. that had a non-None grad).

In [ ]:
def ex2_grad_sync_skip_none(rank: int, world_size: int, dist_module, model: 'nn.Module') -> int:
    """Sync grads across ranks, skipping params with grad is None. Return synced count."""
    raise NotImplementedError()


def _test_ex2():

    import threading
    import contextlib
    import types as _types
    from unittest.mock import patch
    import torch as _t_for_fake
    import torch.distributed as _dist_real

    class _FakeReduceOp:
        SUM = 'SUM'
        MAX = 'MAX'
        MIN = 'MIN'
        PRODUCT = 'PROD'

    class _FakeWorld:
        """Shared state across `world_size` rank-threads."""
        def __init__(self, world_size):
            self.world_size = world_size
            self.barrier = threading.Barrier(world_size)
            self.lock = threading.Lock()
            # scratch[op_id] -> list of (rank, tensor); reset per op via barrier
            self.scratch = {}
            # per-rank thread-local pinned rank
            self.tls = threading.local()
            # collected per-rank results (for the test to read)
            self.results = [None] * world_size
        def all_reduce(self, tensor, op='SUM'):
            rank = self.tls.rank
            # phase 1: every rank deposits its tensor copy
            self.barrier.wait()
            with self.lock:
                self.scratch.setdefault('ar', [None] * self.world_size)
                self.scratch['ar'][rank] = tensor.detach().clone()
            self.barrier.wait()
            # phase 2: every rank reads-out the reduced result (same math)
            bag = self.scratch['ar']
            if op == 'SUM':
                reduced = bag[0].clone()
                for x in bag[1:]:
                    reduced = reduced + x
            elif op == 'MAX':
                reduced = bag[0].clone()
                for x in bag[1:]:
                    reduced = _t_for_fake.maximum(reduced, x)
            elif op == 'MIN':
                reduced = bag[0].clone()
                for x in bag[1:]:
                    reduced = _t_for_fake.minimum(reduced, x)
            elif op == 'PROD':
                reduced = bag[0].clone()
                for x in bag[1:]:
                    reduced = reduced * x
            else:
                raise ValueError(f'unknown fake op {op!r}')
            # mutate in-place so caller's tensor reflects the reduction
            tensor.copy_(reduced)
            self.barrier.wait()
            if rank == 0:
                self.scratch.pop('ar', None)
            self.barrier.wait()
        def reduce(self, tensor, dst, op='SUM'):
            rank = self.tls.rank
            self.barrier.wait()
            with self.lock:
                self.scratch.setdefault('rd', [None] * self.world_size)
                self.scratch['rd'][rank] = tensor.detach().clone()
            self.barrier.wait()
            # only the dst rank gets the reduced result
            if rank == dst:
                bag = self.scratch['rd']
                if op == 'SUM':
                    reduced = bag[0].clone()
                    for x in bag[1:]:
                        reduced = reduced + x
                elif op == 'MAX':
                    reduced = bag[0].clone()
                    for x in bag[1:]:
                        reduced = _t_for_fake.maximum(reduced, x)
                elif op == 'MIN':
                    reduced = bag[0].clone()
                    for x in bag[1:]:
                        reduced = _t_for_fake.minimum(reduced, x)
                elif op == 'PROD':
                    reduced = bag[0].clone()
                    for x in bag[1:]:
                        reduced = reduced * x
                else:
                    raise ValueError(f'unknown fake op {op!r}')
                tensor.copy_(reduced)
            self.barrier.wait()
            if rank == 0:
                self.scratch.pop('rd', None)
            self.barrier.wait()
        def broadcast(self, tensor, src):
            rank = self.tls.rank
            self.barrier.wait()
            if rank == src:
                with self.lock:
                    self.scratch['bc'] = tensor.detach().clone()
            self.barrier.wait()
            if rank != src:
                tensor.copy_(self.scratch['bc'])
            self.barrier.wait()
            if rank == 0:
                self.scratch.pop('bc', None)
            self.barrier.wait()
        def barrier_op(self):
            self.barrier.wait()

    def _run_fake_world(worker_fn, world_size, *extra_args, timeout=30):
        world = _FakeWorld(world_size)
        errors = [None] * world_size
        def _runner(rank):
            world.tls.rank = rank
            # Build the fake `dist` module facade.
            fake_dist = _types.SimpleNamespace()
            fake_dist.ReduceOp = _FakeReduceOp
            fake_dist.all_reduce = lambda tensor, op='SUM': world.all_reduce(tensor, op)
            fake_dist.reduce = lambda tensor, dst, op='SUM': world.reduce(tensor, dst, op)
            fake_dist.broadcast = lambda tensor, src: world.broadcast(tensor, src)
            fake_dist.barrier = world.barrier_op
            fake_dist.get_rank = lambda: rank
            fake_dist.get_world_size = lambda: world_size
            fake_dist.init_process_group = lambda **kw: None
            fake_dist.destroy_process_group = lambda: None
            # Inject into the worker's calling globals.
            # The student code calls `dist.<op>`; we patch the `dist` name
            # in the calling namespace via direct globals injection.
            try:
                worker_fn(rank, world_size, fake_dist, world)
            except BaseException as e:
                import traceback as _tb
                errors[rank] = (e, _tb.format_exc())
        threads = [threading.Thread(target=_runner, args=(r,), daemon=True) for r in range(world_size)]
        for th in threads:
            th.start()
        for th in threads:
            th.join(timeout=timeout)
        for r, err in enumerate(errors):
            if err is not None:
                raise RuntimeError(f'rank {r} failed: {err[0]!r}\n{err[1]}')
        return world.results


    import torch.nn as nn

    # Model with two params; rank-specific grads.
    # Param `active` always gets a grad. Param `frozen` is left at grad=None.
    class _TwoParam(nn.Module):
        def __init__(self):
            super().__init__()
            self.active = nn.Parameter(t.zeros(3))
            self.frozen = nn.Parameter(t.zeros(3))

    def _worker(rank, world_size, dist_module, world):
        model = _TwoParam()
        # Each rank gets a DIFFERENT grad on `active`.
        model.active.grad = t.tensor([float(rank + 1)] * 3)
        # `frozen.grad` stays None — simulates frozen layer / sparse head.
        assert model.frozen.grad is None, 'precondition: frozen.grad starts None'
        synced = ex2_grad_sync_skip_none(rank, world_size, dist_module, model)
        world.results[rank] = (synced, model.active.grad.tolist(), model.frozen.grad)

    # 3 ranks: active.grad = [1,1,1], [2,2,2], [3,3,3] → mean [2,2,2]. frozen stays None.
    results = _run_fake_world(_worker, 3)
    for rank, res in enumerate(results):
        assert res is not None, f'rank {rank} returned None'
        synced, active_grad, frozen_grad = res
        assert synced == 1, f'rank {rank}: expected 1 param synced (only active), got {synced}'
        expected = [2.0, 2.0, 2.0]
        for i, (a, b) in enumerate(zip(active_grad, expected)):
            assert abs(a - b) < 1e-5, f'rank {rank} active.grad[{i}]: got {a}, expected {b}'
        assert frozen_grad is None, f'rank {rank}: frozen.grad must remain None (was skipped), got {frozen_grad}'

    # 2 ranks, all params have grad — synced count = 2.
    def _worker_all_grads(rank, world_size, dist_module, world):
        model = _TwoParam()
        model.active.grad = t.tensor([float(rank + 1)] * 3)
        model.frozen.grad = t.tensor([float((rank + 1) * 10)] * 3)
        synced = ex2_grad_sync_skip_none(rank, world_size, dist_module, model)
        world.results[rank] = (synced, model.active.grad.tolist(), model.frozen.grad.tolist())

    results_all = _run_fake_world(_worker_all_grads, 2)
    for rank, res in enumerate(results_all):
        synced, active_grad, frozen_grad = res
        assert synced == 2, f'rank {rank}: expected 2 params synced, got {synced}'
        # active: [1,1,1] and [2,2,2] → mean [1.5,1.5,1.5]
        for v in active_grad:
            assert abs(v - 1.5) < 1e-5
        # frozen: [10,10,10] and [20,20,20] → mean [15,15,15]
        for v in frozen_grad:
            assert abs(v - 15.0) < 1e-5

    # All params have grad None → synced count = 0, no all_reduce called, no crash.
    def _worker_no_grads(rank, world_size, dist_module, world):
        model = _TwoParam()
        # Both stay None.
        synced = ex2_grad_sync_skip_none(rank, world_size, dist_module, model)
        world.results[rank] = (synced, model.active.grad, model.frozen.grad)

    results_none = _run_fake_world(_worker_no_grads, 3)
    for rank, res in enumerate(results_none):
        synced, a, f = res
        assert synced == 0, f'rank {rank}: expected 0 synced, got {synced}'
        assert a is None and f is None, 'grads must stay None when skipped'
    _dd_passed.add('ex2')
    print("ex2 ✓")

_test_ex2()

<details><summary>Solution</summary>

```python
def ex2_grad_sync_skip_none(rank: int, world_size: int, dist_module, model: 'nn.Module') -> int:
    synced = 0
    for p in model.parameters():
        if p.grad is None:
            continue
        dist_module.all_reduce(p.grad, op=dist_module.ReduceOp.SUM)
        p.grad /= world_size
        synced += 1
    return synced
```

**Why `is None`, not `== 0`.** A zero-VALUED grad tensor still needs to be reduced — it's just one rank's contribution that happens to be zero. `None` means 'no grad was computed this step' (e.g. backward was never called for this param's subgraph). Only the latter should be skipped.

**Rank consistency is load-bearing.** If rank 0 thinks `p.grad is None` and rank 1 thinks it has a grad, rank 1 calls `all_reduce` with no counterpart — deadlock. In practice every rank runs the same forward/backward on the same model graph, so the `is None` answer is identical. The invariant is preserved by the data-parallelism contract, not by any check in this function.

**Real DDP fuses + skips automatically.** `torch.nn.parallel.DistributedDataParallel` hooks backward and kicks an all_reduce when each param's grad becomes ready — skipping frozen layers naturally. The hand-rolled loop you're writing is the conceptual model; DDP is the optimized production version.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()